# 04 — Generation Evaluation (End-to-End)

A reproducible, end-to-end notebook that exercises the **entire pipeline**:
1. (Re-)generate the demo PDFs
2. Ingest them into Chroma
3. Run retrieval, Q&A, summary, and MCQ evaluations
4. Report all rubric metrics: **Hit@3, Hit@5, QA Accuracy, ROUGE-L, BLEU, latency**

**Prereqs**: `ollama serve` running and `llama3.1:8b` pulled.

Run all cells top-to-bottom.

In [ ]:
import sys, time, json
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd

from app.config import get_settings
from app.services.embedding_service import Embedder
from app.services.pdf_loader import load_pdf_pages
from app.services.text_chunker import chunk_pages
from app.services.vector_store import ChromaStore
from app.services.rag_service import answer_question, summarize
from app.services.retrieval_service import retrieve
from app.services.mcq_service import generate_mcqs

settings = get_settings()
print('LLM:', settings.active_llm_model, '@', settings.active_llm_base_url)
print('Embedding:', settings.embedding_model)
print('Chunk size / overlap:', settings.chunk_size, '/', settings.chunk_overlap)
print('Top-K:', settings.top_k)

## Step 1 — Generate demo PDFs (idempotent)

In [ ]:
import subprocess
result = subprocess.run(
    [sys.executable, '../scripts/generate_demo_pdfs.py'],
    capture_output=True, text=True, cwd='..'
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)

## Step 2 — Ingest PDFs into Chroma

In [ ]:
PDF_DIR = Path('../data/raw/uploaded_pdfs')
pdfs = sorted(PDF_DIR.glob('*.pdf'))
total = 0
for p in pdfs:
    pages = load_pdf_pages(p)
    chunks = chunk_pages(pages, settings.chunk_size, settings.chunk_overlap)
    if not chunks:
        print(f'skip {p.name}')
        continue
    embeddings = Embedder.embed_texts([c.text for c in chunks])
    n = ChromaStore.add_chunks(chunks, embeddings)
    print(f'{p.name}: indexed {n} chunks')
    total += n
print(f'\nTotal chunks in store: {ChromaStore.count()}')

## Step 3 — Retrieval evaluation

Computes Hit@3, Hit@5, MRR, Precision@5.

In [ ]:
ret_df = pd.read_csv('../data/evaluation/retrieval_eval_set.csv')
TOP_K = 5
ret_rows = []
for _, row in ret_df.iterrows():
    t0 = time.perf_counter()
    hits = retrieve(row['query'], top_k=TOP_K)
    latency = time.perf_counter() - t0
    matches = [
        (h.file_name == row['relevant_file']) and (h.page_number == int(row['relevant_page']))
        for h in hits
    ]
    rr = next((1.0/i for i, m in enumerate(matches, 1) if m), 0.0)
    ret_rows.append({
        'hit@3': int(any(matches[:3])),
        'hit@5': int(any(matches)),
        'precision@5': sum(matches)/TOP_K,
        'mrr': rr,
        'latency_s': latency,
    })
ret_summary = pd.DataFrame(ret_rows).mean().to_dict()
ret_summary

## Step 4 — Q&A evaluation (ROUGE-L, BLEU, QA Accuracy)

In [ ]:
from rouge_score import rouge_scorer
from sacrebleu import sentence_bleu

rouge = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

def qa_accuracy(reference: str, hypothesis: str) -> int:
    ref_tokens = {t.lower().strip('.,?!:;') for t in reference.split() if len(t) > 3}
    hyp_lower = hypothesis.lower()
    if not ref_tokens:
        return 0
    hits = sum(1 for t in ref_tokens if t in hyp_lower)
    return 1 if hits / len(ref_tokens) >= 0.5 else 0

In [ ]:
qa_df = pd.read_csv('../data/evaluation/qa_eval_set.csv')
qa_rows = []
for _, row in qa_df.iterrows():
    t0 = time.perf_counter()
    answer, sources = answer_question(row['query'])
    latency = time.perf_counter() - t0
    ref = row['reference_answer']
    qa_rows.append({
        'qa_accuracy': qa_accuracy(ref, answer),
        'rouge_l': rouge.score(ref, answer)['rougeL'].fmeasure,
        'bleu': sentence_bleu(answer, [ref]).score / 100.0,
        'grounded': int(bool(sources)),
        'latency_s': latency,
    })
qa_summary = pd.DataFrame(qa_rows).mean().to_dict()
qa_summary

## Step 5 — Summary evaluation (ROUGE-1/2/L)

In [ ]:
rouge_all = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
sum_df = pd.read_csv('../data/evaluation/summary_eval_set.csv')
sum_rows = []
for _, row in sum_df.iterrows():
    t0 = time.perf_counter()
    summary, sources = summarize(row['query'])
    latency = time.perf_counter() - t0
    scores = rouge_all.score(row['reference_summary'], summary)
    sum_rows.append({
        'rouge_1': scores['rouge1'].fmeasure,
        'rouge_2': scores['rouge2'].fmeasure,
        'rouge_l': scores['rougeL'].fmeasure,
        'grounded': int(bool(sources)),
        'latency_s': latency,
    })
summary_summary = pd.DataFrame(sum_rows).mean().to_dict()
summary_summary

## Step 6 — MCQ evaluation

In [ ]:
mcq_df = pd.read_csv('../data/evaluation/mcq_eval_set.csv')
mcq_rows = []
for _, row in mcq_df.iterrows():
    t0 = time.perf_counter()
    items = generate_mcqs(row['topic'], int(row['num_questions']), row['difficulty'])
    latency = time.perf_counter() - t0
    format_ok = sum(1 for q in items if q.correct_answer in {'A','B','C','D'}) / max(len(items), 1)
    distinct = sum(
        len({q.choices.A.strip().lower(), q.choices.B.strip().lower(), q.choices.C.strip().lower(), q.choices.D.strip().lower()}) / 4.0
        for q in items
    ) / max(len(items), 1)
    mcq_rows.append({
        'count': len(items),
        'format_ok': format_ok,
        'distinct_choices': distinct,
        'latency_s': latency,
    })
mcq_summary = pd.DataFrame(mcq_rows).mean().to_dict()
mcq_summary

## Final report

In [ ]:
report = {
    'retrieval': ret_summary,
    'qa': qa_summary,
    'summary': summary_summary,
    'mcq': mcq_summary,
}
print(json.dumps(report, indent=2, default=float))